# Separately trained one-factor controls

Formal DiffJev workflow; see AGENTS.md and prompt.md.

In [ ]:
import os, sys, gc
from pathlib import Path
ROOT = Path(os.environ.get('DIFFJEV_ROOT', Path.cwd())).resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'AGENTS.md').exists() and Path('/content/drive/MyDrive/DiffJev/AGENTS.md').exists():
    ROOT = Path('/content/drive/MyDrive/DiffJev')
assert (ROOT / 'AGENTS.md').exists(), 'Mount Drive and set DIFFJEV_ROOT to the checkout used by notebook 00'
sys.path.insert(0, str(ROOT / 'src'))
from diffjev.notebook_runtime import NotebookRuntime
runtime = NotebookRuntime(ROOT)
runtime.python_code("from diffjev.preparation import check_environment; from pathlib import Path; check_environment(Path.cwd())", label='06_exp4_ablation_environment')
def run_python(source):
    prefix = '''import gc
from pathlib import Path
from diffjev import workflows as w
from diffjev.config import load_yaml_config
ROOT = w.ROOT
cfg = load_yaml_config(str(ROOT / 'configs/workflow.yaml'))
SEED, DEVICE = cfg['seed'], cfg['device']
'''
    runtime.python_code(prefix + source, label='06_exp4_ablation')
print('Experiment worker:', runtime.python)


In [ ]:
run_python(r"""
for variant in cfg['anchor_ablations']:
    print(w.train_run('llada_8b_base',variant=variant,seed=SEED,device=DEVICE))
    gc.collect()
""")

In [ ]:
run_python(r"""
for variant in w.CORE_ABLATIONS:
    w.evaluate_ablation(variant,seed=SEED,device=DEVICE)
    gc.collect()
w.collect_results()
""")

The T=1 control has full own-detail access in its only pass, no corruption loss, and no iterative feedback. No-pruning reuses the full trained model. No-augmentation replays the same source trace with native menus. No factorial grid or extra experiment is introduced.